# Feature Track 2: Reliable & Structured Outputs

Building a RAG system that returns text is easy; building one that returns **predictable, machine-readable data** is what makes it production-ready.

In this notebook, we move beyond free-form chat. We want our RAG pipeline to extract specific attributes—like GWP values, certification dates, and supplier names—into a strict schema that can be used by other software or stored in a database without manual parsing.

### The Problem: The "Wall of Text"
In the previous stages, the LLM might provide the correct answer, but it is often buried in prose. For a compliance dashboard or an automated verification system, we do not want a paragraph; we want specific fields like product IDs and numeric values extracted accurately.

### Goals for this Notebook
1. **Schema Definition:** Define our data requirements.
2. **Constrained Generation:** Force the LLM to adhere to the schema using "Structured Outputs".
3. **Data Validation:** Implement runtime checks to ensure extracted numbers and dates are within logical bounds.
4. **Failure Handling:** Gracefully manage cases where the required information is missing from the retrieved context.

In [ ]:
import os
import pathlib
import warnings

from conversational_toolkit.agents.base import QueryWithContext
from conversational_toolkit.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
)
from conversational_toolkit.embeddings.openai import (
    OpenAIEmbeddings,
)
from conversational_toolkit.vectorstores.chromadb import ChromaDBVectorStore

from sme_kt_zh_collaboration_rag.feature0_baseline_rag import (
    EMBEDDING_MODEL,
    VS_PATH,
    SYSTEM_PROMPT,
    load_chunks,
    build_vector_store,
    build_llm,
    build_agent,
)
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=DeprecationWarning)

load_dotenv(dotenv_path="../../.env.local")

_secret_path = pathlib.Path("/secrets/OPENAI_API_KEY")
if "OPENAI_API_KEY" not in os.environ and _secret_path.exists():
    os.environ["OPENAI_API_KEY"] = _secret_path.read_text().strip()

RETRIEVER_TOP_K = 5
BACKEND = "openai"  # "ollama" or "openai"
FORCE_REBUILD = False  # set True to re-embed from scratch and rebuild the vector store

if not BACKEND:
    raise ValueError('Set BACKEND to "ollama" or "openai" before running.')

# RAG pipeline
if "sentence-transformers" in EMBEDDING_MODEL:
    embedding_model = SentenceTransformerEmbeddings(model_name=EMBEDDING_MODEL)
else:
    embedding_model = OpenAIEmbeddings(model_name=EMBEDDING_MODEL)

if FORCE_REBUILD or not VS_PATH.exists():
    chunks = load_chunks()
    vs = await build_vector_store(
        chunks, embedding_model, db_path=VS_PATH, reset=FORCE_REBUILD
    )
else:
    # Vector store already exists -> open it without re-embedding
    vs = ChromaDBVectorStore(db_path=str(VS_PATH))
    print(
        f"Reusing existing vector store at {VS_PATH} ({vs.collection.count()} chunks)"
    )

llm = build_llm(backend=BACKEND)
agent = build_agent(
    vector_store=vs,
    embedding_model=embedding_model,
    llm=llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=SYSTEM_PROMPT,
    number_query_expansion=0,
)

print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Vector store: {VS_PATH}")
print(f"RAG agent LLM: {BACKEND}")
print("Setup complete.")

In [ ]:
query = "Does PrimePack AG offer a product called the Lara Pallet?"

response = await agent.answer(QueryWithContext(query=query, history=[]))

print(f"Query: {query}")
print("-" * 20)
print(response.content)

In [ ]:
# Redefining the System Prompt to explicitly request Source IDs
SYSTEM_PROMPT_WITH_SOURCES = (
    "You are a helpful AI assistant specialised in sustainability and product compliance for PrimePack AG.\n\n"
    "Each context chunk is prefixed with [ID: ...].\n"
    "Rules:\n"
    "- Use only the provided excerpts.\n"
    "- For every fact, include the ID in parentheses, e.g., (ID: 123).\n"
    "- List all unique IDs in a 'Sources' section at the end.\n"
    "- If unknown, say so clearly."
)

# Re-building the agent with the updated prompt
agent = build_agent(
    vector_store=vs,
    embedding_model=embedding_model,
    llm=llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=SYSTEM_PROMPT_WITH_SOURCES,
    number_query_expansion=0,
)

# Test query
query = "Which products have a third-party verified EPD?"
response = await agent.answer(QueryWithContext(query=query, history=[]))
print(response.content)

In [ ]:
# Define the JSON Schema
compliance_schema = {
    "name": "ProductCompliance",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "products": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "product_name": {"type": "string"},
                        "has_verified_epd": {"type": "boolean"},
                        "co2_kg_co2e": {"type": ["number", "null"]},
                        "confidence_score": {"type": "number"},
                    },
                    "required": [
                        "product_name",
                        "has_verified_epd",
                        "co2_kg_co2e",
                        "confidence_score",
                    ],
                    "additionalProperties": False,
                },
            },
            "referenced_ids": {"type": "array", "items": {"type": "string"}},
        },
        "required": ["products", "referenced_ids"],
        "additionalProperties": False,
    },
}

# Configure the LLM with the native response_format
structured_llm = build_llm(
    backend=BACKEND,
    response_format={"type": "json_schema", "json_schema": compliance_schema},
)

agent = build_agent(
    vector_store=vs,
    embedding_model=embedding_model,
    llm=structured_llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=SYSTEM_PROMPT_WITH_SOURCES,
    number_query_expansion=0,
)

query = "Which products have a third-party verified EPD?"
response = await agent.answer(QueryWithContext(query=query, history=[]))
print(response.content)